In [ ]:
!pip install pytest ipytest

In [21]:
import pytest
import ipytest
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from unittest.mock import MagicMock, patch

In [28]:
ipytest.autoconfig()

In [22]:
DATASET_PATH = Path.cwd() / "models" / "empid_test_set.csv"
MODEL_PATH = Path.cwd() / "models" / "xgb_model.joblib"

In [23]:
ALL_COLS = [
    "EmployeeID", "Age", "BusinessTravel", "DailyRate", "DistanceFromHome",
    "Education", "EducationField", "EnvironmentSatisfaction", "HourlyRate",
    "JobInvolvement", "JobLevel", "JobRole", "JobSatisfaction", "MaritalStatus",
    "MonthlyIncome", "MonthlyRate", "NumCompaniesWorked", "OverTime",
    "PercentSalaryHike", "PerformanceRating", "RelationshipSatisfaction", "Shift",
    "TotalWorkingYears", "TrainingTimesLastYear", "WorkLifeBalance",
    "YearsAtCompany", "YearsInCurrentRole", "YearsSinceLastPromotion",
    "YearsWithCurrManager", "Department_Cardiology", "Department_Maternity",
    "Department_Neurology", "Gender_Female", "Gender_Male",
    "is_other_JobRole", "Attrition",
]

FEATURE_COLS  = [c for c in ALL_COLS if c not in ("EmployeeID", "Attrition")]
GROUP_COL     = "JobRole"
CLASS_NAMES   = ["No Attrition", "Attrition"]

In [24]:
# Real EmployeeIDs present in the dataset
KNOWN_EMP_IDS  = [1628396, 1658398, 1520532, 1410589, 1309082] # randomly selected from the dataset.
EMP_HIGH_OVERTIME = 1126499 # overtime = 1, which is the highest correlation with attrition (r=0.38)
MISSING_EMP_ID = 9999999   # guaranteed absent

Fixtures

In [25]:
from src.functions import get_employee_shap_values, explainn

In [26]:
@pytest.fixture(scope="session")
def dataset():
    return pd.read_csv(DATASET_PATH)

@pytest.fixture(scope="session")
def model():
    return joblib.load(MODEL_PATH)

@pytest.fixture(scope="session")
def X_test_df(dataset):
    return dataset[FEATURE_COLS].copy()

@pytest.fixture
def mock_model():
    m = MagicMock()
    m.predict_proba = MagicMock(
        side_effect=lambda X: np.column_stack(
            [np.full(len(X), 0.5), np.full(len(X), 0.5)]
        )
    )
    return m

@pytest.fixture
def shap_result():
    return get_employee_shap_values(model(), EMP_HIGH_OVERTIME, dataset())

Test for Local

In [27]:
class TestGlobalReturnStructure:
    def test_overtime_drives_attrition(shap_result):
        assert shap_result["OverTime"] > 0

    def test_low_job_satisfaction_drives_attrition(shap_result):
        assert shap_result["JobSatisfaction"] > 0

    def test_low_work_life_balance_drives_attrition(shap_result):
        assert shap_result["WorkLifeBalance"] > 0



In [29]:
ipytest.run()

FFF                                                                                          [100%]
============================================ FAILURES =============================================
____________________ TestGlobalReturnStructure.test_overtime_drives_attrition _____________________

shap_result = <__main__.TestGlobalReturnStructure object at 0x000001C85AC5AD90>

    def test_overtime_drives_attrition(shap_result):
>       assert shap_result["OverTime"] > 0
               ^^^^^^^^^^^^^^^^^^^^^^^
E       TypeError: 'TestGlobalReturnStructure' object is not subscriptable

C:\Users\HP\AppData\Local\Temp\ipykernel_29948\2357735862.py:3: TypeError
______________ TestGlobalReturnStructure.test_low_job_satisfaction_drives_attrition _______________

shap_result = <__main__.TestGlobalReturnStructure object at 0x000001C85B4F3B10>

    def test_low_job_satisfaction_drives_attrition(shap_result):
>       assert shap_result["JobSatisfaction"] > 0
               ^^^^^^^^^^^^^^^^^^^^^

<ExitCode.TESTS_FAILED: 1>